In [2]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive.zip


In [3]:
import zipfile

zip_file = "archive.zip"   # Replace with your ZIP file name

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

In [4]:
import os

os.listdir('/content/dataset')

['ml-100k']

In [7]:
os.listdir("dataset")

['ml-100k']

In [8]:
!pip install scikit-surprise -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install numpy==1.26.4 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 

In [1]:
!pip install numpy==1.26.4 --force-reinstall

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.

In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae

In [3]:
import os

os.listdir("dataset")

['ml-100k']

In [4]:
import os

os.listdir("dataset/ml-100k")

['u5.base',
 'u.item',
 'ua.base',
 'allbut.pl',
 'u2.base',
 'u.data',
 'mku.sh',
 'u1.test',
 'u.info',
 'u5.test',
 'u4.test',
 'u.occupation',
 'README',
 'ub.test',
 'ub.base',
 'u2.test',
 'ua.test',
 'u3.base',
 'u3.test',
 'u1.base',
 'u.user',
 'u.genre',
 'u4.base']

In [5]:
df = pd.read_csv(
    "dataset/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

df.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [6]:
df = df.drop("timestamp", axis=1)
df.head()

,user_id,movie_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [7]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    df[['user_id', 'movie_id', 'rating']],
    reader
)

In [8]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [9]:
model = SVD()
model.fit(trainset)

In [10]:
predictions = model.test(testset)

In [11]:
print("RMSE:")
rmse(predictions)

print("\nMAE:")
mae(predictions)

RMSE:
RMSE: 0.9386

MAE:
MAE:  0.7383


0.738290346401642

In [12]:
user_id = 1
movie_id = 50

pred = model.predict(user_id, movie_id)

print("Predicted rating:", pred.est)

Predicted rating: 4.913154286957851


In [13]:
def recommend(user_id, n=5):

    all_movies = df['movie_id'].unique()
    rated = df[df['user_id'] == user_id]['movie_id'].tolist()

    unrated = [m for m in all_movies if m not in rated]

    predictions = []

    for movie in unrated:
        pred = model.predict(user_id, movie)
        predictions.append((movie, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTop {n} movies for user {user_id}:")

    for movie, rating in predictions[:n]:
        print(movie, round(rating, 2))

recommend(1)


Top 5 movies for user 1:
408 4.88
525 4.76
313 4.72
488 4.7
641 4.61
